Load the dataset

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('Cars Datasets 2025.csv', encoding="latin1")
df.describe(include="all")
df.info() 

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 11 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   Company Names              1218 non-null   object
 1   Cars Names                 1218 non-null   object
 2   Engines                    1218 non-null   object
 3   CC/Battery Capacity        1215 non-null   object
 4   HorsePower                 1218 non-null   object
 5   Total Speed                1218 non-null   object
 6   Performance(0 - 100 )KM/H  1212 non-null   object
 7   Cars Prices                1218 non-null   object
 8   Fuel Types                 1218 non-null   object
 9   Seats                      1218 non-null   object
 10  Torque                     1217 non-null   object
dtypes: object(11)
memory usage: 104.8+ KB


Initial data quality check

In [2]:
# Missing values
df.isna().sum()

Company Names                0
Cars Names                   0
Engines                      0
CC/Battery Capacity          3
HorsePower                   0
Total Speed                  0
Performance(0 - 100 )KM/H    6
Cars Prices                  0
Fuel Types                   0
Seats                        0
Torque                       1
dtype: int64

In [3]:
# Percent missing values
(df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

Performance(0 - 100 )KM/H    0.492611
CC/Battery Capacity          0.246305
Torque                       0.082102
Engines                      0.000000
Cars Names                   0.000000
Company Names                0.000000
HorsePower                   0.000000
Total Speed                  0.000000
Cars Prices                  0.000000
Fuel Types                   0.000000
Seats                        0.000000
dtype: float64

In [4]:
# Duplicates
df.duplicated().sum()

np.int64(4)

In [5]:
# Unique values
for col in df.columns:
    print(f'\n{col}')
    print(df[col].unique()[:20])


Company Names
['FERRARI' 'ROLLS ROYCE' 'Ford' 'MERCEDES' 'AUDI' 'BMW' 'ASTON MARTIN'
 'BENTLEY' 'LAMBORGHINI' 'TOYOTA' 'NISSAN' 'ROLLS ROYCE ' 'VOLVO' 'KIA'
 'HONDA' 'KIA  ' 'HYUNDAI' 'MAHINDRA' 'MARUTI SUZUKI' 'Nissan']

Cars Names
['SF90 STRADALE' 'PHANTOM' 'KA+' ' GT 63 S' 'AUDI R8 Gt' 'Mclaren 720s'
 'VANTAGE F1' 'Continental GT Azure' 'VENENO ROADSTER' 'F8 TRIBUTO'
 '812 GTS' 'PORTOFINO' 'ROMA' 'MONZA SP2' 'F8 SPIDER' 'PORTOFINO M'
 'ROMA SPIDER' 'GR SUPRA' 'TOYOTA 86' 'TOYOTA  GR86']

Engines
['V8' 'V12' '1.2L Petrol' 'V10' 'I4' 'BOXER-4' 'V6' 'ELECTRIC MOTOR' 'I6'
 'ELECTRIC ' 'ELECTRIC' 'I3' 'I4 + ELECTRIC' 'HYBRID'
 '1.2L,4-CYLINDER,INLINE-4(I4)' '1.4L,4-CYLINDER,INLINE-4(I4)'
 '2.0L,4-CYLINDER,INLINE-4(I4)' '2.2L,4-CYLINDER,INLINE-4(I4)'
 '1.5L,4-CYLINDER,INLINE(I4)' '2.0L,4-CYLINDER,WITH HYBRID SYSTEM']

CC/Battery Capacity
['3990 cc' '6749 cc' '1,200 cc' '3,982 cc' '5,204 cc' '3,994 cc'
 '3,996 cc' '6,498 cc' '3,900 cc' '6496 cc' '6,496 cc' '2,998 cc'
 '1,998 cc' '2,387 cc

In [6]:
# Convert common representations of missing values to NaN
missing_values = ['-', '--', 'missing', 'null', 'NULL', 'NA', 'N/A', '', 'nan', 'N/A (Concept Only)']
df.replace(missing_values, pd.NA, inplace=True)
df.isna().sum()

Company Names                0
Cars Names                   0
Engines                      0
CC/Battery Capacity          5
HorsePower                   0
Total Speed                  0
Performance(0 - 100 )KM/H    6
Cars Prices                  1
Fuel Types                   0
Seats                        0
Torque                       1
dtype: int64

Data Cleaning

In [7]:
# renaming column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("(", "")
    .str.replace(")", "")
    .str.replace("/", "_")
)
df.columns

Index(['company_names', 'cars_names', 'engines', 'cc_battery_capacity',
       'horsepower', 'total_speed', 'performance0_-_100_km_h', 'cars_prices',
       'fuel_types', 'seats', 'torque'],
      dtype='object')

In [8]:
# clean numerical values
df['horsepower'].unique()
df["horsepower"] = (
    df["horsepower"]
    .astype(str)
    .str.replace("hp", "", regex=False)
    .str.replace("HP", "", regex=False)
    .str.replace("cc", "", regex=False)
    .str.replace("(est.)", "", regex=False)
    .str.replace("\x96", "-", regex=False)
    .str.replace("Up to", "", regex=False)
    .str.replace("~", "", regex=False)
    .str.strip()
)
def convert_range(value):
    value = str(value).replace(",", "").strip()
    
    if "-" in value:
        parts = value.split("-")
        try:
            return (float(parts[0]) + float(parts[1])) / 2
        except:
            return np.nan
    try:
        return float(value)
    except:
        return np.nan
df["horsepower"] = df["horsepower"].apply(convert_range) 
df['horsepower'].isna().sum()

np.int64(6)

In [9]:
# Clean torque
df['torque'].unique()  
df['torque'] = (
    df['torque']
    .astype(str)
    .str.replace('Nm', "", regex=False)
    .str.replace('(GT Model)', '', regex=False)
    .str.replace('(Electric)', '', regex=False)
    .str.replace('\x96', '-',regex=False)
    .str.replace('/', '-', regex=False)
    .str.strip()
)  
df["torque"] = df["torque"].apply(convert_range) 
df['torque'].isna().sum()


np.int64(3)

In [10]:
# Clean top speed
df['total_speed'].unique()
df['total_speed'] = (
    df["total_speed"]
    .astype(str)
    .str.replace("km/h", "", regex=False)
    .str.replace('(est.)', '', regex=False)
    .str.strip()
) 
df["total_speed"] = df["total_speed"].apply(convert_range)
df['total_speed'].unique()

array([340., 250., 165., 320., 341., 314., 318., 356., 226., 220., 200.,
       315., 290., 225., 240., 180., 402., 362., 322., 313., 291., 328.,
       324., 350., 325., 305., 355., 285., 280., 330., 230., 213., 216.,
       185., 195., 211., 210., 272., 201., 186., 209., 190., 160., 170.,
       175., 120., 130., 215., 140., 235., 233., 205., 183., 145., 238.,
       196., 217., 138., 148., 193., 191., 222., 246., 187., 237., 168.,
       270., 155., 354., 110., 150., 289., 300., 245., 265., 286., 295.,
       232., 259., 293., 308., 306., 296., 260., 275., 302., 301., 310.,
       253., 263., 221., 242., 304., 311., 307., 105., 162., 125., 261.,
       262., 177., 312., 420., 490., 380., 500., 100.,  90.,  80., 283.,
       348., 204.,  85.])

In [11]:
# clean engine
df['cc_battery_capacity'].unique() 
df["engine_cc"] = (
    df["cc_battery_capacity"]
    .astype(str)
    .str.extract(r"([\d,.]+)\s*cc", expand=False)
)

df["engine_cc"] = (
    df["engine_cc"]
    .str.replace(",", "", regex=False)
    .astype(float)
)
df["battery_kwh"] = (
    df["cc_battery_capacity"]
    .astype(str)
    .str.extract(r"([\d,.]+)\s*kwh", expand=False)
)

df["battery_kwh"] = (
    df["battery_kwh"]
    .str.replace(",", "", regex=False)
    .astype(float)
)
df["has_battery"] = df["battery_kwh"].notna().astype(int)
df["has_engine"] = df["engine_cc"].notna().astype(int)
df["engine_cc"] = df["engine_cc"].fillna(0)
df["battery_kwh"] = df["battery_kwh"].fillna(0)
df["battery_kwh"].unique()

array([ 0.  , 95.  , 11.6 , 13.8 ,  1.49, 64.8 ,  1.24, 53.6 , 39.2 ,
        1.56])

In [12]:
# Creating another features
df["vehicle_type"] = np.select(
    [
        (df["has_engine"] == 1) & (df["has_battery"] == 1),
        (df["has_engine"] == 1) & (df["has_battery"] == 0),
        (df["has_engine"] == 0) & (df["has_battery"] == 1)
    ],
    [
        "Hybrid",
        "ICE",
        "Electric"
    ],
    default="Unknown"
)

In [13]:
# clean price
df['cars_prices'].unique()[:30]
df['cars_prices'] = (
    df['cars_prices']
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace("\x96", "-", regex=False)
    .str.replace("\x80", "-", regex=False)
    .str.replace(',', "", regex=False)
    .str.strip()    
) 
def extract_price(value):
    value = str(value).replace("$", "").replace(",", "").strip()
    
    if "-" in value:
        parts = value.split("-")
        
        try:
            low = float(parts[0])
            high = float(parts[1])
            return (low + high) / 2
        except:
            return np.nan
    
    try:
        return float(value)
    except:
        return np.nan
df['cars_prices']= df['cars_prices'].apply(extract_price)
df['cars_prices']= df['cars_prices'].fillna(df['cars_prices'].median())
df['cars_prices'].unique()

array([1.10000e+06, 4.60000e+05, 1.35000e+04, 1.61000e+05, 2.53290e+05,
       4.99000e+05, 1.93440e+05, 3.11000e+05, 4.50000e+06, 2.80000e+05,
       3.50000e+05, 2.10000e+05, 2.30000e+05, 1.70000e+06, 2.20000e+05,
       2.40000e+05, 5.39000e+04, 2.70000e+04, 3.00000e+04, 8.50000e+04,
       5.00000e+04, 1.13000e+05, 4.00000e+04, 2.50000e+04, 3.50000e+04,
       2.00000e+04, 2.80000e+04, 3.20000e+06, 3.16000e+05, 2.08000e+05,
       1.42000e+05, 1.89000e+05, 2.94000e+05, 1.30000e+06, 2.80000e+06,
       5.18000e+05, 2.74000e+05, 2.61000e+05, 4.93000e+05, 2.11000e+05,
       2.87000e+05, 4.45000e+05, 3.08000e+05, 4.21000e+05, 2.42000e+05,
       5.45000e+05, 2.58000e+05, 5.73000e+05, 2.63000e+05, 3.27000e+05,
       2.73000e+05, 6.03000e+05, 2.53000e+05, 3.42000e+05, 3.32000e+05,
       3.30000e+05, 3.60000e+05, 3.25000e+05, 5.00000e+05, 3.70000e+05,
       3.90000e+05, 4.50000e+05, 3.40000e+05, 3.80000e+05, 3.20000e+05,
       1.16000e+05, 1.04000e+05, 5.30000e+04, 1.09000e+05, 6.200

In [14]:
# clean acceleration
df['performance0_-_100_km_h'].unique()
df["acceleration_0_100"] = (
    df["performance0_-_100_km_h"]
    .astype(str)
    .str.extract(r"([\d.]+)")[0] 
    .astype(float)
)
df = df.drop(columns=["performance0_-_100_km_h"])
df["acceleration_0_100"].unique() 

array([ 2.5,  5.3, 10.5,  3.2,  3.6,  2.9,  4. ,  3.4,  4.1,  6.4,  5.6,
        6.7,  6.9,  4.7,  7.3,  5.8,  8.2,  7.5,  6.5,  6.8,  5.9,  3.9,
        4.5,  4.2,  2.8,  3. ,  3.1,  3.3,  4.8,  4.4,  4.9,  5.2,  4.3,
        5.1,  5.5,  5.7,  6.1,  3.7,  3.5,  5.4,  6.3,  3.8,  8.5, 10.3,
       10.9,  8.4,  7.1, 12.2, 11.2,  8.9,  7.7,  7.2,  8. ,  8.3,  7.9,
        8.1,  6. ,  9.4,  9.5,  9.2,  6.2, 10.2,  7.6, 11.5,  5. ,  9. ,
       10. ,  nan, 11. , 14.8, 13.5,  9.3,  7.4, 14. ,  9.8,  7.8, 13. ,
        7. ,  4.6,  8.8, 14.3, 12.9, 10.8, 23. , 17.5, 11.7, 10.4,  8.6,
       12.4,  9.6,  9.9,  8.7, 12. , 11.9, 14.5, 10.7, 10.6,  6.6, 10.1,
       13.7, 12.5, 15. , 12.3, 12.8, 18. , 18.5, 13.8,  2.7,  2.6, 29. ,
       16. , 16.5, 17. , 17.6,  2.1,  1.9,  9.1,  2.3,  2.4,  2.2, 20. ,
       22. ,  9.7, 15.5, 35. ])

In [15]:
# Handling missing values
df.isnull().sum()
# For numerical columns
numeric_cols = df.select_dtypes(include=np.number).columns

df[numeric_cols] = df[numeric_cols].fillna(
    df[numeric_cols].median()
)

# For categorical columns
categorical_cols = df.select_dtypes(include="object").columns

for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
df.isnull().sum()

company_names          0
cars_names             0
engines                0
cc_battery_capacity    0
horsepower             0
total_speed            0
cars_prices            0
fuel_types             0
seats                  0
torque                 0
engine_cc              0
battery_kwh            0
has_battery            0
has_engine             0
vehicle_type           0
acceleration_0_100     0
dtype: int64

In [16]:
# drop duplicates
df.duplicated().sum()
df = df.drop_duplicates()
df.shape

(1213, 16)

In [17]:
df.info()
df.duplicated().sum()
df.describe()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
Index: 1213 entries, 0 to 1217
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   company_names        1213 non-null   object 
 1   cars_names           1213 non-null   object 
 2   engines              1213 non-null   object 
 3   cc_battery_capacity  1213 non-null   object 
 4   horsepower           1213 non-null   float64
 5   total_speed          1213 non-null   float64
 6   cars_prices          1213 non-null   float64
 7   fuel_types           1213 non-null   object 
 8   seats                1213 non-null   object 
 9   torque               1213 non-null   float64
 10  engine_cc            1213 non-null   float64
 11  battery_kwh          1213 non-null   float64
 12  has_battery          1213 non-null   int64  
 13  has_engine           1213 non-null   int64  
 14  vehicle_type         1213 non-null   object 
 15  acceleration_0_100   1213 non-null   float6

company_names          0
cars_names             0
engines                0
cc_battery_capacity    0
horsepower             0
total_speed            0
cars_prices            0
fuel_types             0
seats                  0
torque                 0
engine_cc              0
battery_kwh            0
has_battery            0
has_engine             0
vehicle_type           0
acceleration_0_100     0
dtype: int64

In [18]:
# Extraction of csv file
df.to_csv('final_cleaned_data.csv', index=False, encoding='utf-8')
